In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import time
import torch.nn.functional as F
import timm
from sklearn.metrics import accuracy_score, classification_report
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf

IMAGE_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


test_ds      = datasets.ImageFolder("test", transform=transform)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


num_classes = len(test_ds.classes)  

model_path = 'models/InceptionV3.keras'
try:
    modelk = tf.keras.models.load_model(model_path)
    print("Model loaded successfully!")
    modelk.summary()
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure the path to your .keras file is correct and TensorFlow is installed.")

    

predictionresult = []
def inference(model1, loader):
    y_true, y_pred = [], []
    class_names = loader.dataset.classes  
    with torch.no_grad():
        for batch_idx, (imgs_batch, labels_batch) in enumerate(loader):
            
            for i in range(len(imgs_batch)):
                img_single = imgs_batch[i].unsqueeze(0).to(device)  # (1, C, H, W)
                label_single = labels_batch[i].item()
                keras_input = img_single.permute(0, 2, 3, 1).cpu().numpy()
                strt = time.perf_counter()
                predictions_keras = model1.predict(keras_input, verbose=0)
                out1 = torch.from_numpy(predictions_keras).to(device)
                prob1 = F.softmax(out1, dim=1)
                pred_single = prob1.argmax(dim=1)
                y_true.append(label_single)
                y_pred.append(pred_single.item())

    return y_true, y_pred


y_true, y_pred = inference(
    modelk,
    test_loader
)
acc = accuracy_score(y_true, y_pred)
print("Ensemble Accuracy:", acc)

print(classification_report(
    y_true,
    y_pred,
    target_names=test_ds.classes
))

/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
2026-07-20 18:14:57.903173: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different c

Model loaded successfully!


/home/ai_atrlbcau/miniconda3/envs/tensor_torch/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 382 variables whereas the saved optimizer has 386 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 299, 299,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 149, 149,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 149, 149,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 149, 149,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 147, 147,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 147, 147,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 147, 147,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 147, 147,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 73, 73,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 73, 73,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 73, 73,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 73, 73,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 71, 71,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 71, 71,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 71, 71,    │          0 │ batch_normalizat

...

 Trainable params: 21,782,695 (83.09 MB)

 Non-trainable params: 34,432 (134.50 KB)

 Optimizer params: 43,565,392 (166.19 MB)

I0000 00:00:1784551502.721372  409420 service.cc:152] XLA service 0x7765580554c0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1784551502.721403  409420 service.cc:160]   StreamExecutor device (0): Host, Default Version
2026-07-20 18:15:02.777579: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1784551503.584054  409420 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Execution Time: 2.572966 seconds
Execution Time: 0.922114 seconds
Execution Time: 0.929967 seconds
Execution Time: 1.289384 seconds
Execution Time: 0.924623 seconds
Execution Time: 1.291342 seconds
Execution Time: 0.930019 seconds
Execution Time: 0.923732 seconds
Execution Time: 0.960203 seconds
Execution Time: 0.951812 seconds
Execution Time: 0.947506 seconds
Execution Time: 0.934912 seconds
Execution Time: 0.948177 seconds
Execution Time: 0.942040 seconds
Execution Time: 0.945199 seconds
Execution Time: 0.948689 seconds
Execution Time: 0.958328 seconds
Execution Time: 0.970029 seconds
Execution Time: 0.957323 seconds
Execution Time: 0.974833 seconds
Execution Time: 1.289459 seconds
Execution Time: 0.956667 seconds
Execution Time: 1.290097 seconds
Execution Time: 0.947348 seconds
Execution Time: 0.970587 seconds
Execution Time: 0.975197 seconds
Execution Time: 1.286496 seconds
Execution Time: 0.978304 seconds
Execution Time: 0.981308 seconds
Execution Time: 1.298518 seconds
Execution 